# 🏠 House Price Prediction — Advanced Version
Kaggle competition: **House Prices - Advanced Regression Techniques**

Flow: EDA → Cleaning → Feature Engineering → Preprocessing Pipeline → Multi-model CV → Blend → Save pipeline → Submission

This notebook is designed to run directly on **Kaggle** (dataset pre-attached) but also works locally if you place `train.csv`/`test.csv` in a `data/` folder and adjust the paths in the next cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import xgboost as xgb
import lightgbm as lgb

import joblib

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42


In [ ]:
# --- 1. Load data ---
# On Kaggle, the dataset is auto-attached under /kaggle/input/
# Locally, point these at your data/ folder instead.

KAGGLE_PATH = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/"
LOCAL_PATH = "../data/"

import os
DATA_PATH = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution — SalePrice is right-skewed, which is why we'll log-transform it
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("SalePrice (raw)")
sns.histplot(np.log1p(train["SalePrice"]), kde=True, ax=axes[1])
axes[1].set_title("SalePrice (log1p transformed)")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation with SalePrice — which raw features matter most?
numeric_df = train.select_dtypes(include=np.number)
corr = numeric_df.corr()["SalePrice"].sort_values(ascending=False)
corr.head(15)


In [ ]:
# Missing values overview
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})


In [ ]:
# Outlier check — GrLivArea vs SalePrice (a well-known outlier pair in this dataset)
plt.figure(figsize=(8, 5))
sns.scatterplot(data=train, x="GrLivArea", y="SalePrice")
plt.title("GrLivArea vs SalePrice (look for the 2 large-area, low-price outliers)")
plt.show()


## 3. Data Cleaning — Outliers

Remove the two well-known outliers: very large living area but unusually low price
(these are almost certainly data entry quirks, not representative houses).

In [ ]:
train = train[~((train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000))].reset_index(drop=True)
print("Train shape after outlier removal:", train.shape)


## 4. Feature Engineering + Missing Value Handling

Using the shared `src/features.py` module so the training notebook and the deployed
Streamlit app apply *exactly* the same logic. If running on Kaggle where `src/` isn't
available, the functions are redefined inline below (safe fallback).

In [ ]:
# Feature engineering functions (self-contained — no external src/ import needed,
# so this notebook runs standalone on Kaggle with zero setup).
# This is the SAME logic as src/features.py in the GitHub repo, kept in sync manually.

NONE_MEANS_ABSENT = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]

ZERO_MEANS_ABSENT = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea",
]

QUALITY_MAP = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ORDINAL_QUALITY_COLS = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
    "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC",
]


def handle_missing_values(df):
    df = df.copy()
    for col in NONE_MEANS_ABSENT:
        if col in df.columns:
            df[col] = df[col].fillna("None")
    for col in ZERO_MEANS_ABSENT:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    if "LotFrontage" in df.columns and "Neighborhood" in df.columns:
        df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
            lambda x: x.fillna(x.median())
        )
    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].mode()[0])
    num_cols = df.select_dtypes(include=np.number).columns
    for col in num_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    return df


def engineer_features(df):
    df = df.copy()
    df["TotalSF"] = df.get("TotalBsmtSF", 0) + df.get("1stFlrSF", 0) + df.get("2ndFlrSF", 0)
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]
    df["IsRemodeled"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)
    df["TotalBath"] = (
        df.get("FullBath", 0) + 0.5 * df.get("HalfBath", 0)
        + df.get("BsmtFullBath", 0) + 0.5 * df.get("BsmtHalfBath", 0)
    )
    df["TotalPorchSF"] = (
        df.get("OpenPorchSF", 0) + df.get("EnclosedPorch", 0)
        + df.get("3SsnPorch", 0) + df.get("ScreenPorch", 0)
    )
    df["HasPool"] = (df.get("PoolArea", 0) > 0).astype(int)
    df["HasGarage"] = (df.get("GarageArea", 0) > 0).astype(int)
    df["HasBasement"] = (df.get("TotalBsmtSF", 0) > 0).astype(int)
    df["HasFireplace"] = (df.get("Fireplaces", 0) > 0).astype(int)
    return df


def encode_ordinal_quality(df):
    df = df.copy()
    for col in ORDINAL_QUALITY_COLS:
        if col in df.columns:
            df[col] = df[col].map(QUALITY_MAP).fillna(0).astype(int)
    return df


def full_feature_pipeline(df):
    df = handle_missing_values(df)
    df = engineer_features(df)
    df = encode_ordinal_quality(df)
    return df


print("Feature engineering functions defined.")


In [ ]:
# Separate target, keep Id for submission, combine train+test for consistent processing
train_ids = train["Id"]
test_ids = test["Id"]

y = np.log1p(train["SalePrice"])  # log-transform target (fixes the skew we saw in EDA)

train_features = train.drop(columns=["Id", "SalePrice"])
test_features = test.drop(columns=["Id"])

# Combine so categorical encoding sees the same categories on both sides
all_features = pd.concat([train_features, test_features], axis=0, ignore_index=True)

all_features = full_feature_pipeline(all_features)

# Log-transform skewed numeric features (not just the target)
from scipy.stats import skew
numeric_feats = all_features.select_dtypes(include=np.number).columns
skewed_feats = all_features[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
high_skew = skewed_feats[abs(skewed_feats) > 0.75].index
for feat in high_skew:
    all_features[feat] = np.log1p(all_features[feat].clip(lower=0))

print("Skewed numeric features log-transformed:", len(high_skew))

# Split back into train/test
X = all_features.iloc[:len(train_features), :].reset_index(drop=True)
X_test = all_features.iloc[len(train_features):, :].reset_index(drop=True)

print("X shape:", X.shape, "| X_test shape:", X_test.shape)


## 5. Preprocessing Pipeline

Numeric columns → median-impute + scale. Categorical (nominal) columns → most-frequent-impute + one-hot.
This whole thing gets wrapped into the final model Pipeline so it travels together as one saved object.

In [ ]:
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include="object").columns.tolist()

print(f"{len(numeric_cols)} numeric columns, {len(categorical_cols)} categorical columns")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])


## 6 & 7. Cross-Validation Setup + Train Multiple Models

We compare Ridge, Random Forest, XGBoost, and LightGBM using 5-Fold CV on RMSE
(of the log-transformed target — this matches Kaggle's actual competition metric).

In [ ]:
def rmse_cv(model, X, y, folds=5):
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    kf = KFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipe, X, y, scoring="neg_root_mean_squared_error", cv=kf)
    return -scores.mean(), -scores.std()

models = {
    "Ridge": Ridge(alpha=10.0, random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=1000, learning_rate=0.03, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03, num_leaves=20,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    ),
}

results = {}
for name, model in models.items():
    mean_rmse, std_rmse = rmse_cv(model, X, y)
    results[name] = mean_rmse
    print(f"{name:15s} RMSE: {mean_rmse:.4f} (+/- {std_rmse:.4f})")


## 8. Model Comparison

In [ ]:
results_df = pd.DataFrame(results.items(), columns=["Model", "CV_RMSE"]).sort_values("CV_RMSE")
results_df


In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=results_df, x="CV_RMSE", y="Model", palette="viridis")
plt.title("Model comparison (lower RMSE = better)")
plt.tight_layout()
plt.show()


## 9. Blend the Top Models

Instead of picking a single "best" model, we fit the top performers on the full
training data and blend their predictions with weights (more weight to the
stronger models). This almost always outperforms any single model.

In [ ]:
# Fit each model's full pipeline on all training data
fitted_pipelines = {}
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X, y)
    fitted_pipelines[name] = pipe
    print(f"Fitted: {name}")


In [ ]:
# Weighted blend — adjust weights based on your own CV results above.
# A simple starting point: weight inversely proportional to each model's RMSE.
BLEND_WEIGHTS = {
    "Ridge": 0.15,
    "RandomForest": 0.15,
    "XGBoost": 0.35,
    "LightGBM": 0.35,
}

def blended_predict(X_input):
    preds = np.zeros(len(X_input))
    for name, weight in BLEND_WEIGHTS.items():
        preds += weight * fitted_pipelines[name].predict(X_input)
    return preds

# Sanity check on training data (not a true held-out score, just a smoke test)
train_preds_log = blended_predict(X)
train_rmse = np.sqrt(mean_squared_error(y, train_preds_log))
print(f"Blended RMSE on training data: {train_rmse:.4f}")


## 10. Final Prediction + Kaggle Submission

In [ ]:
test_preds_log = blended_predict(X_test)
test_preds = np.expm1(test_preds_log)  # reverse the log1p transform

submission = pd.DataFrame({"Id": test_ids, "SalePrice": test_preds})
submission.to_csv("submission.csv", index=False)
submission.head()


## Save the pipeline for deployment

We save the **best single model's pipeline** (simplest to deploy and load in Streamlit).
If you want the blended ensemble in production too, save `fitted_pipelines` as a dict
instead and replicate `blended_predict` inside the app — the single-best-model version
below is the simpler, recommended starting point.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = fitted_pipelines[best_model_name]
print(f"Saving best single model: {best_model_name}")

joblib.dump(best_pipeline, "house_price_pipeline.joblib")
print("Saved house_price_pipeline.joblib — download this from the notebook Output panel")
print("and place it in your local project's models/ folder.")


### Next steps
1. Download `house_price_pipeline.joblib` and `submission.csv` from this notebook's Output panel
2. Place the `.joblib` file in `models/` in your local `house-price-prediction` project
3. Run the Streamlit app locally (`streamlit run app/app.py`) to test predictions
4. Push everything to GitHub
5. Deploy for free on Streamlit Community Cloud (share.streamlit.io)

### Optional upgrades (see README.md for details)
- Add SHAP explainability to the app
- Swap RandomizedSearchCV/manual tuning for Optuna
- Track experiments with MLflow
- Add pytest tests + a GitHub Actions CI workflow